In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
train = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/train.csv')
test = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv').drop(columns=['id'])

In [4]:
train['humidity'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['humidity'].to_numpy()])
train['pressure'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['pressure'].to_numpy()])
train['wind_speed'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['wind_speed'].to_numpy()])

test['humidity'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['humidity'].to_numpy()])
test['pressure'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['pressure'].to_numpy()])
test['wind_speed'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['wind_speed'].to_numpy()])

In [5]:
column = list(train.columns)
column.remove('efficiency')
column.remove('id')
column.remove('string_id')
column.remove('error_code')
column.remove('installation_type')

for i in column[:6]:
    for j in column[6:]:
        print(i,j)
        if i != j:
            train[f'{i} x {j}'] = (train[i].astype('float').to_numpy() * train[j].astype('float').to_numpy())
            test[f'{i} x {j}'] = (test[i].astype('float').to_numpy() * test[j].astype('float').to_numpy())

temperature voltage
temperature current
temperature module_temperature
temperature cloud_coverage
temperature wind_speed
temperature pressure
irradiance voltage
irradiance current
irradiance module_temperature
irradiance cloud_coverage
irradiance wind_speed
irradiance pressure
humidity voltage
humidity current
humidity module_temperature
humidity cloud_coverage
humidity wind_speed
humidity pressure
panel_age voltage
panel_age current
panel_age module_temperature
panel_age cloud_coverage
panel_age wind_speed
panel_age pressure
maintenance_count voltage
maintenance_count current
maintenance_count module_temperature
maintenance_count cloud_coverage
maintenance_count wind_speed
maintenance_count pressure
soiling_ratio voltage
soiling_ratio current
soiling_ratio module_temperature
soiling_ratio cloud_coverage
soiling_ratio wind_speed
soiling_ratio pressure


In [6]:
train

,id,temperature,irradiance,humidity,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,...,maintenance_count x module_temperature,maintenance_count x cloud_coverage,maintenance_count x wind_speed,maintenance_count x pressure,soiling_ratio x voltage,soiling_ratio x current,soiling_ratio x module_temperature,soiling_ratio x cloud_coverage,soiling_ratio x wind_speed,soiling_ratio x pressure
0,0,7.817315,576.179270,41.243087,32.135501,4.0,0.803199,37.403527,1.963787,13.691147,...,54.764589,249.976178,51.299648,4075.466021,30.042481,1.577312,10.996718,50.195163,10.300958,818.352702
1,1,24.785727,240.003973,1.359648,19.977460,8.0,0.479456,21.843315,0.241473,27.545096,...,220.360769,350.809906,96.096349,8204.990830,10.472905,0.115776,13.206657,21.024731,5.759244,491.741324
2,2,46.652695,687.612799,91.265368,1.496401,4.0,0.822398,48.222882,4.191800,43.363708,...,173.454832,NaN,7.257599,4043.690616,39.658424,3.447330,35.662247,NaN,1.492160,831.381250
3,3,53.339567,735.141179,96.190955,18.491582,3.0,0.837529,46.295748,0.960567,57.720436,...,173.161309,202.084418,26.208777,3065.539990,38.774015,0.804502,48.342518,56.417163,7.316867,855.825843
4,4,5.575374,12.241203,27.495073,30.722697,6.0,0.551833,0.000000,0.898062,6.786263,...,40.717576,21.792000,3.136103,6051.335747,0.000000,0.495580,3.744882,2.004256,0.288434,556.554179
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,19995,16.868428,NaN,93.530318,14.393967,3.0,0.738911,12.147711,3.005355,26.206810,...,78.620429,5.199040,37.782367,3055.123401,8.976080,2.220690,19.364505,1.280543,9.305938,752.488281
19996,19996,53.415061,296.970303,93.985714,25.997012,2.0,0.513061,0.000000,0.532119,65.000000,...,130.000000,129.117334,1.953982,2032.162204,0.000000,0.273009,33.348965,33.122534,0.501256,521.311588
19997,19997,2.442727,660.328019,37.968918,32.818396,9.0,0.548602,13.047950,4.075498,11.584869,...,104.263818,519.571202,42.758435,9087.160153,7.158128,2.235825,6.355479,31.670849,2.606372,553.914609
19998,19998,NaN,632.760700,43.014702,19.063517,4.0,NaN,0.000000,1.068906,21.149351,...,84.597402,312.494756,45.216634,4026.695498,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [8]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].mean(), inplace=True)
        train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].mean(), inplace=True)
        test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [9]:
train_x = train.drop(columns=['id', 'efficiency'])
train_y = train['efficiency']

In [10]:
LGBM_R_parm = {'colsample_bytree': 0.6657998165699125,
               'drop_rate': 0.8303473371870002,
               'learning_rate': 0.2762739125344964,
               'max_bin': 9983,
               'max_depth': 8623,
               'max_drop': 5480,
               'min_child_samples': 101,
               'min_data_in_leaf': 426,
               'n_estimators': 1628,
               'num_leaves': 3640,
               'objective': 'regression_l1',
               'reg_alpha': 0.39940189926691194,
               'reg_lambda': 0.9567353299218986,
               'skip_drop': 0.6274640597528743,
               'verbosity': -1}

XGB_R_parm = {'colsample_bytree': 0.871893537724492,
              'gamma': 1,
              'learning_rate': 0.923192518624813,
              'max_depth': 15,
              'min_child_weight': 1,
              'n_estimators': 17748,
              'objective': 'binary:logistic',
              'reg_alpha': 45,
              'reg_lambda': 0.8598696247943665,
              'subsample': 0.9109367356405415}

catboost_params = {'iterations' : 1000,
                   'learning_rate': 0.009, 
                   'depth': 5, 
                   'l2_leaf_reg': 5.5,
                   'min_child_samples' : 102,
                   'od_wait' : 50,
                   'random_state' : 42,
                   'eval_metric': 'RMSE', 
                   'bootstrap_type': 'Bayesian', 
                   'grow_policy' : 'Depthwise',
                   'logging_level' : 'Silent'
                 }

X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.1, random_state=100)

In [11]:
LGBM_R = LGBMRegressor(**LGBM_R_parm)

In [12]:
XGB_R = XGBRegressor(**XGB_R_parm)

In [13]:
catboost = CatBoostRegressor(learning_rate=0.009,
                             depth=5,
                             l2_leaf_reg=5.5,
                             min_child_samples=102,
                             grow_policy='Depthwise',
                             iterations=1000,
                             eval_metric='RMSE',
                             od_type='Iter',
                             od_wait=50,
                             random_state=42,
                             logging_level='Silent')

In [14]:
estimators = [
    ('LGBM', make_pipeline(SimpleImputer(), LGBM_R)),
    ('xgb', make_pipeline(SimpleImputer(), XGB_R)),
    ('cat', make_pipeline(SimpleImputer(), catboost)),
    ('RF', make_pipeline(SimpleImputer(), RandomForestRegressor(n_estimators=25, 
                                                                 min_samples_split=2, 
                                                                 max_depth=10, 
                                                                 min_samples_leaf=2, 
                                                                 random_state=0))),
    ('ET', make_pipeline(SimpleImputer(), ExtraTreesRegressor(n_estimators=50, 
                                                               max_depth=8, 
                                                               min_samples_split=2, 
                                                               min_samples_leaf=2, 
                                                               random_state=42)))
]
model = StackingRegressor(estimators, final_estimator = RidgeCV())
model.fit(X_train, y_train)

StackingRegressor(estimators=[('LGBM',
                               Pipeline(steps=[('simpleimputer',
                                                SimpleImputer()),
                                               ('lgbmregressor',
                                                LGBMRegressor(colsample_bytree=0.6657998165699125,
                                                              drop_rate=0.8303473371870002,
                                                              learning_rate=0.2762739125344964,
                                                              max_bin=9983,
                                                              max_depth=8623,
                                                              max_drop=5480,
                                                              min_child_samples=101,
                                                              min_data_in_leaf=426,
                                                              n_estimators=1628,
                                                              num_leaves=3640,
                                                              objective='reg...
                               Pipeline(steps=[('simpleimputer',
                                                SimpleImputer()),
                                               ('randomforestregressor',
                                                RandomForestRegressor(max_depth=10,
                                                                      min_samples_leaf=2,
                                                                      n_estimators=25,
                                                                      random_state=0))])),
                              ('ET',
                               Pipeline(steps=[('simpleimputer',
                                                SimpleImputer()),
                                               ('extratreesregressor',
                                                ExtraTreesRegressor(max_depth=8,
                                                                    min_samples_leaf=2,
                                                                    n_estimators=50,
                                                                    random_state=42))]))],
                  final_estimator=RidgeCV())

In [15]:
print(f'Concordance index (lifelines): {100*(1-numpy.sqrt(mean_squared_error(y_test, model.predict(X_test))))}')

Concordance index (lifelines): 90.26862970298244


In [16]:
id = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv')['id']
test_predictions = model.predict(test)
test_predictions

array([0.38581717, 0.53430609, 0.51716677, ..., 0.62055364, 0.42650585,
       0.55737692])

In [17]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions,
})
submission

,id,efficiency
0,0,0.385817
1,1,0.534306
2,2,0.517167
3,3,0.459814
4,4,0.479774
...,...,...
11995,11995,0.542319
11996,11996,0.463467
11997,11997,0.620554
11998,11998,0.426506


In [18]:
submission.to_csv('submission.csv', index = False)